Hadoop Ecosystem    

Hadoop is an open-source framework for distributed storage and processing of large datasets across clusters of commodity hardware.

## Core Hadoop Components

<img src="./pic/2_HADOOP-ECOSYSTEM.png" width=700>

- Processing Layer: MapReduce / Spark / Hive / Pig
- Resource Management: YARN (ResourceManager, NodeManager, ApplicationMaster)
- Storage Layer: HDFS (NameNode, DataNode)
- Data Ingestion: Kafka, Flume, Sqoop



## HDFS (Hadoop Distributed File System)

### Why HDFS?

**Problem**: How do you store a 1TB log file?
- Single machine can't store it safely
- Single machine isn't reliable (hardware failures)
- Single machine can't read it fast enough

**Solution**: HDFS **splits, distributes, and replicates data**


### Core Design Principles

```text
       Original large File (1TB)
              │
              ▼
       ┌──────────────────────────────────────────────────────────────┐
       │        1. Split into Blocks (default 128 MB each)            │
       │  [Block 1] [Block 2] [Block 3] ... [Block 8192]              │
       └──────────────────────────────────────────────────────────────┘
              │
              ▼
       ┌────────────────────────────────────────────────────────────────┐
       │     2. Blocks are distribute Across DataNodes (machines)       │
       │                                                                │
       │  DataNode 1    DataNode 2    DataNode 3    DataNode 4          │
       │  ┌─────────┐   ┌─────────┐   ┌─────────┐   ┌─────────┐         │
       │  │Block 1  │   │Block 1  │   │Block 2  │   │Block 1  │ ←Replica│
       │  │Block 2  │   │Block 3  │   │Block 3  │   │Block 4  │         │
       │  │Block 5  │   │Block 4  │   │Block 6  │   │Block 5  │         │
       │  └─────────┘   └─────────┘   └─────────┘   └─────────┘         │
       │  3. Each block is stored with multiple replicas (default: 3)   │
       └────────────────────────────────────────────────────────────────┘
```



#### Three Key Roles:   

**1. NameNode (Master)**

| Responsibility | Description |
|---------------|-------------|
| Metadata Management | Tracks which blocks make up each file |
| Block Location | Knows where each block is stored |
| Namespace Management | Maintains file system directory tree |
| Client Coordination | Directs clients to appropriate DataNodes |


NameNode Metadata Example:
```text
/user/data/sales.csv
├── Block 1 → [DataNode1, DataNode3, DataNode5]
├── Block 2 → [DataNode2, DataNode4, DataNode6]
└── Block 3 → [DataNode1, DataNode2, DataNode4]
```

**2. DataNode (Worker)**

| Responsibility | Description |
|---------------|-------------|
| Block Storage | Stores actual data blocks |
| Block Operations | Handles read/write requests |
| Heartbeat | Reports status to NameNode |
| Replication | Participates in block replication |

**3. Client**

- Applications that read/write data (**Spark, MapReduce, Hive**)
- Communicates with NameNode for metadata
- Communicates with DataNodes for actual data



#### NameNode VS DataNode 

| Aspect | NameNode | DataNode |
|------|---------|----------|
| Role | Master / Controller | Worker / Storage |
| Stores actual data | No | Yes |
| Stores metadata | Yes | No |
| Metadata includes | File names, directory structure, permissions, block locations | Block IDs only |
| Client interaction | Provides block locations to clients | Serves data directly to clients |
| Failure impact | Critical (cluster becomes unavailable) | Non-critical (data replicated elsewhere) |
| Scalability | Vertical | Horizontal |
| Typical count | 1 active (+ standby) | Many (hundreds or thousands) |



#### HDFS Configuration Defaults

| Parameter | Default Value | Description |
|-----------|---------------|-------------|
| Block Size | 128 MB | Size of each data block |
| Replication Factor | 3 | Number of copies of each block |
| Heartbeat Interval | 3 seconds | DataNode health check frequency |



## YARN (Yet Another Resource Negotiator)

### What Does YARN Do?

YARN is the **resource management** layer of Hadoop. It decides **who** runs what, 
**where** it runs, and with **how many** resources:    
- Tracks available machines in the cluster
- Manages CPU and memory resources
- Schedules jobs
- Monitors execution and retries failed tasks



### YARN Components

```text
┌─────────────────────────────────────────────────────────────────────┐
│                          YARN Architecture                          │
│  ┌────────────────────────────────────────────────────────────────┐ │
│  │                    ResourceManager (RM)                        │ │
│  │                    [Global Authority]                          │ │
│  │    • Manages cluster-wide resources                            │ │
│  │    • Accepts job submissions                                   │ │
│  │    • Schedules applications                                    │ │
│  └────────────────────────────────────────────────────────────────┘ │
│                              │                                      │
│              ┌───────────────┼───────────────┐                      │
│              ▼               ▼               ▼                      │
│  ┌──────────────────┐ ┌──────────────────┐ ┌──────────────────┐     │
│  │  NodeManager 1   │ │  NodeManager 2   │ │  NodeManager 3   │     │
│  │  [Worker Node]   │ │  [Worker Node]   │ │  [Worker Node]   │     │
│  │                  │ │                  │ │                  │     │
│  │  ┌────────────┐  │ │  ┌────────────┐  │ │  ┌────────────┐  │     │
│  │  │ Container  │  │ │  │ Container  │  │ │  │ Container  │  │     │
│  │  │  (App      │  │ │  │  (Task)    │  │ │  │  (Task)    │  │     │
│  │  │  Master)   │  │ │  │            │  │ │  │            │  │     │
│  │  └────────────┘  │ │  └────────────┘  │ │  └────────────┘  │     │
│  └──────────────────┘ └──────────────────┘ └──────────────────┘     │
└─────────────────────────────────────────────────────────────────────┘
```



#### Component Responsibilities

| Component | Scope | Responsibilities |
|-----------|-------|------------------|
| **ResourceManager (RM)** | Cluster-wide | Global resource allocation, job scheduling |
| **NodeManager (NM)** | Per-machine | Reports local resources, manages containers |
| **ApplicationMaster (AM)** | Per-application | Requests resources, coordinates tasks |
| Container | Per-task | Isolated execution environment |



### YARN Job Execution Flow

<img src="./pic/2_Job-execution-process-in-YARN.png" width=700>



1. <font color=brown>Client</font> **submits application** to <font color=orange>ResourceManager</font>,   

   <font color=orange>ResourceManager</font> **allocates container** for <font color=blue>ApplicationMaster</font>

2. <font color=blue>ApplicationMaster</font> starts and **registers** with <font color=orange>ResourceManager</font>

3. <font color=blue>ApplicationMaster</font> **requests containers** for tasks

4. <font color=orange>ResourceManager</font> **allocates containers** on <font color=green>NodeManagers</font>

5. <font color=blue>ApplicationMaster</font> **launches tasks** in containers,    

   **Tasks execute and report progress** to <font color=blue>ApplicationMaster</font>

6. Application **completes**, <font color=blue>ApplicationMaster</font> **deregisters**




## MapReduce vs Spark

### MapReduce (First Generation Hadoop compute engine)

**Characteristics**:
- **Disk-based** execution model
- Data written to disk after each stage
- High latency due to I/O overhead
- Reliable and scalable
- **Best For**: **Long-running batch jobs** where latency isn't critical

**Execution Pattern**:
```text
Input → Map → [Disk] → Shuffle → [Disk] → Reduce → Output
                ↑                    ↑
           Disk I/O            Disk I/O
```


**Example**: Word Count in MapReduce
```text
Map Phase:    "hello world" → [("hello", 1), ("world", 1)]
Shuffle:      Group by key
Reduce Phase: [("hello", [1,1,1])] → [("hello", 3)]
```



### Spark (Next Generation Hadoop compute engine)

Apache Spark running on a Hadoop cluster (specifically using YARN as its cluster manager and often HDFS for storage).   

Designed to replace MapReduce for most workloads

**Characteristics**:
- **In-memory** execution
- DAG-based optimization
- Low latency, interactive analytics
- Unified engine (supports batch, SQL, streaming, ML)
- **Best For**: Interactive analytics, iterative algorithms, real-time processing

**Execution Pattern**:
```text
Input → Transform → Transform → Transform → Output
              ↑           ↑           ↑
           Memory     Memory     Memory
           (cached)   (cached)   (cached)
```



### Comparison Table

| Aspect | MapReduce | Spark |
|--------|-----------|-------|
| Execution Model | Disk-based | In-memory |
| Speed | Slower | 10-100x faster |
| Ease of Use | Complex (Java) | Simple (Python/Scala/SQL) |
| Iterative Algorithms | Poor | Excellent |
| Interactive Queries | Not suitable | Excellent |
| Streaming | Not native | Native support |
| Machine Learning | Limited | MLlib integrated |


